In [1]:
import sys
sys.path.append("..")  # pour trouver le dossier analyse/ depuis notebooks/

import pandas as pd
from analyse.chargement import charger
from analyse.population import preparer

df = preparer(charger(2026))
fr = df[df["Zone"].isin(["France", "Pays frontaliers"])].copy()

# rôles traduits ; « Manager » seul (55 pers.) regroupé avec « Manager qui code »
ROLES = {
    "Individual contributor": "Développeur",
    "Individual contributor in a senior or staff-level role": "Développeur senior",
    "Both people manager and individual contributor": "Manager",
    "People manager": "Manager",
}
fr["Role"] = fr["ICorPM"].map(ROLES)
fr["JobSat"] = pd.to_numeric(fr["JobSat"], errors="coerce")

print(len(fr))
fr["Role"].value_counts()

3376


Role
Développeur           1788
Développeur senior    1076
Manager                496
Name: count, dtype: int64

In [2]:
# satisfaction moyenne par rôle, avec le nombre de répondants
sat = (
    fr.groupby("Role")["JobSat"]
    .agg(satisfaction="mean", repondants="count")
    .round(2)
)
sat

,satisfaction,repondants
Role,,
Développeur,6.87,1775
Développeur senior,7.02,1072
Manager,7.53,495


In [3]:
# satisfaction par rôle, à l'intérieur de chaque tranche d'expérience
moy = fr.pivot_table(index="Tranche_experience", columns="Role",
                     values="JobSat", aggfunc="mean", observed=False).round(2)
nb = fr.pivot_table(index="Tranche_experience", columns="Role",
                    values="JobSat", aggfunc="count", observed=False)
display(moy)
display(nb)

Role,Développeur,Développeur senior,Manager
Tranche_experience,,,
0-1,6.96,8.33,7.50
2-3,6.79,6.67,6.79
4-6,6.74,7.14,6.97
7-10,6.69,6.74,7.11
11-20,6.91,6.97,7.42
21+,7.18,7.20,7.98


Role,Développeur,Développeur senior,Manager
Tranche_experience,,,
0-1,84,3,2
2-3,219,12,14
4-6,320,85,36
7-10,363,180,72
11-20,433,472,198
21+,343,314,170


In [4]:
# même règle que Saad : en stagnation = toute réponse sauf "I do not feel complacent",
# calculée seulement sur ceux qui ont répondu à la question
rep = fr[fr["JobSatComplacent"].notna()].copy()
rep["En_stagnation"] = rep["JobSatComplacent"] != "I do not feel complacent"
rep.groupby("Role")["En_stagnation"].agg(part="mean", repondants="size").round(3)

,part,repondants
Role,,
Développeur,0.605,831
Développeur senior,0.623,528
Manager,0.611,244


In [5]:
# pour chaque tranche d'expérience, la répartition des rôles (chaque ligne fait 100 %)
pd.crosstab(fr["Tranche_experience"], fr["Role"], normalize="index").round(2)


Role,Développeur,Développeur senior,Manager
Tranche_experience,,,
0-1,0.94,0.03,0.02
2-3,0.89,0.05,0.06
4-6,0.72,0.19,0.08
7-10,0.59,0.29,0.12
11-20,0.39,0.43,0.18
21+,0.41,0.38,0.21


In [6]:
FORMATION = {
    "Yes, both on my own and at work": "Seul et au travail",
    "Yes, at work": "Au travail",
    "Yes, on my own": "Seul",
    "No": "Non",
}
fr["Formation"] = fr["LearnAI"].map(FORMATION)
pd.crosstab(fr["Role"], fr["Formation"], normalize="index").round(2)

Formation,Au travail,Non,Seul,Seul et au travail
Role,,,,
Développeur,0.21,0.25,0.15,0.39
Développeur senior,0.21,0.15,0.13,0.51
Manager,0.15,0.14,0.18,0.53


## Ce que je retiens (France et pays frontaliers, 2026, 3 376 répondants)

**Je remarque :**
1. Les managers sont plus satisfaits que les développeurs : 7,53 contre 6,87 sur 10.
2. L'écart reste à expérience égale à partir de 7 ans (+0,4 à +0,8 point). Il ne s'explique donc pas seulement par l'expérience. Avant 7 ans, il y a trop peu de managers pour conclure (moins de 50).
3. Être senior sans encadrer ne change presque rien à la satisfaction.
4. La part en stagnation est la même pour tous les rôles, autour de 61 %. Évoluer ne fait pas disparaître ce sentiment.
5. Entre 2 et 3 ans d'expérience, 89 % sont encore simples développeurs : presque personne n'a évolué au moment où l'ESN les perd.
6. Les simples développeurs sont ceux qui se forment le moins : 25 % ne se sont pas formés à l'IA, contre environ 15 % pour les autres rôles.

**Je me demande :**
- Les managers sont-ils plus satisfaits parce qu'ils encadrent, ou deviennent-ils managers parce qu'ils étaient déjà plus motivés ? L'enquête ne permet pas de trancher.
- Seuls 48 % ont répondu à la question sur la stagnation : le résultat serait-il le même avec tout le monde ?

**Limites :** les répondants sont des volontaires, et l'enquête ne mesure pas les départs.

